# Eclat / Julia — notebook tích hợp A/B

A đã vào main qua PR #1; notebook giữ setup và preparation B cùng miner/benchmark A.
Mặc định chạy preprocessing, miner và kiểm hai backend. Pipeline khai thác ứng dụng
cuối, benchmark tùy chọn và final delivery vẫn cần thực thi/hoàn thiện riêng.
Local run không thay cho Colab cold-run; xem docs/planning/STATUS.md.


In [ ]:
from pathlib import Path
import os
import sys
import hashlib
import re
import shutil
import subprocess
import urllib.request

REPO_URL = "https://github.com/phuctran33/-DM-Eclat.git"
REPO_REF = os.environ.get("ECLAT_REPO_REF", "feat/member-b-applications")
RUN_PREPARATION = True
JULIA_VERSION = "1.11.4"
RUN_MINER = True
RUN_SPMF = False
RUN_OPTIMIZED = True
RUN_APPLICATIONS = False
RUN_BENCHMARKS = False
FINAL_DELIVERY_VALIDATION = False
SEED = 42

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path("/content/eclat-member-b")
    if not (ROOT / "Project.toml").exists():
        if not REPO_URL.startswith("https://github.com/"):
            raise ValueError("Điền URL GitHub public thật của nhóm trước khi chạy Colab.")
        subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(ROOT)], check=True)
    if not shutil.which("julia"):
        archive = Path("/content/julia-runtime.tar.gz")
        runtime = Path("/content/julia-runtime")
        runtime.mkdir(exist_ok=True)
        urllib.request.urlretrieve(
            f"https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-{JULIA_VERSION}-linux-x86_64.tar.gz",
            archive,
        )
        checksum_url = f"https://julialang-s3.julialang.org/bin/checksums/julia-{JULIA_VERSION}.sha256"
        checksum_text = urllib.request.urlopen(checksum_url, timeout=30).read().decode("utf-8")
        filename = f"julia-{JULIA_VERSION}-linux-x86_64.tar.gz"
        expected = next(line.split()[0] for line in checksum_text.splitlines() if line.split()[-1].lstrip("*") == filename)
        assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected, "Julia archive checksum mismatch"
        subprocess.run(["tar", "-xzf", str(archive), "-C", str(runtime), "--strip-components=1"], check=True)
        os.environ["PATH"] = str(runtime / "bin") + os.pathsep + os.environ["PATH"]
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "Project.toml").exists()), None)
    if ROOT is None:
        raise RuntimeError("Mở notebook từ repo root hoặc notebooks/.")

os.chdir(ROOT)
os.environ.setdefault("JULIA_DEPOT_PATH", str(ROOT / ".tools/julia-depot"))
JULIA = os.environ.get("ECLAT_JULIA") or shutil.which("julia")
if not JULIA:
    candidates = sorted((ROOT / ".tools").glob("julia-*/bin/julia.exe"))
    JULIA = str(candidates[-1]) if candidates else None
if not JULIA:
    raise RuntimeError("Cài Julia 1.9+ và thêm vào PATH khi chạy local.")

def run_julia(*arguments, env=None):
    return subprocess.run(
        [JULIA, "--startup-file=no", f"--project={ROOT}", *map(str, arguments)],
        cwd=ROOT, env={**os.environ, **(env or {})}, check=True,
    )

version = subprocess.check_output([JULIA, "--version"], text=True)
match = re.search(r"(\d+)\.(\d+)", version)
assert match and tuple(map(int, match.groups())) >= (1, 9), "Julia 1.9+ required"
assert (ROOT / "scripts/prepare_applications.jl").is_file(), "Checkout phải có code B; chọn đúng nhánh đã publish hoặc mở source B local."
run_julia("--version")
run_julia("-e", "using Pkg; Pkg.instantiate()")
print("ROOT:", ROOT, "SEED:", SEED)
import tempfile
work_parent = ROOT / ".tools/notebook-runs"
work_parent.mkdir(parents=True, exist_ok=True)
NOTEBOOK_OUTPUT = Path(tempfile.mkdtemp(prefix="integrated-", dir=work_parent))


## B: dữ liệu và audit

Nguồn/hash và toàn bộ preprocessing đã được kiểm chéo độc lập. Cell tiếp theo
chuẩn bị dữ liệu thật; dữ liệu này chưa phải kết quả ứng dụng mined cuối cùng.


In [ ]:
if RUN_PREPARATION:
    import tempfile
    subprocess.run([sys.executable, "scripts/prepare_application_data.py", "--dataset", "all"], check=True)
    work_parent = ROOT / ".tools/notebook-runs"
    work_parent.mkdir(parents=True, exist_ok=True)
    B_OUTPUT = Path(tempfile.mkdtemp(prefix="member-b-", dir=work_parent))
    run_julia("scripts/prepare_applications.jl", "groceries", "data/applications/groceries.csv", B_OUTPUT / "groceries")
    run_julia("scripts/prepare_applications.jl", "splice", "data/applications/splice.data", B_OUTPUT / "splice")
    subprocess.run([sys.executable, "-B", "tests/verify_prepared_data.py", "--julia", JULIA,
                    "--report", str(B_OUTPUT / "preparation-report.json")], check=True)
    print("B preprocessing artifacts:", B_OUTPUT)
else:
    print("B preprocessing không chạy trong phiên này.")


## Kiểm thử chung sau tích hợp

Full suite bật miner list/bitset, rules, CLI, SPMF và toàn bộ checks B.


In [ ]:
run_julia("tests/runtests.jl", env={"ECLAT_TEST_MINER": "1", "ECLAT_TEST_SPMF": "1", "ECLAT_TEST_BACKENDS": "list,bitset"})
subprocess.run([sys.executable, "-B", "tests/test_application_sources.py"], check=True)
run_julia("scripts/mine.jl", "--help")


## Miner Basic và references SPMF

CLI chạy code A đã merge. References A/B chạy độc lập và có cùng hashes.
Bật RUN_SPMF nếu cần tái tạo bằng Java >=21; ECLAT_JAVA chọn executable.


In [ ]:
if RUN_MINER:
    run_julia("scripts/mine.jl", "--input", "data/toy/basic.txt", "--output", NOTEBOOK_OUTPUT / "basic.txt",
              "--minsup", "40%", "--backend", "list")
    run_julia("scripts/compare_spmf.jl", NOTEBOOK_OUTPUT / "basic.txt", "tests/reference/spmf/basic.txt")
else:
    print("RUN_MINER đang tắt trong phiên này.")

if RUN_SPMF:
    subprocess.run([sys.executable, "scripts/prepare_references_b.py", "--java", os.environ.get("ECLAT_JAVA", "java")], check=True)
    run_julia("tests/runtests_b.jl")
else:
    print("Giữ 5 references SPMF thật đã pin trong repo; phiên này không tạo lại.")


## Hai ứng dụng

Preprocessing đã có nguồn/hash/mapping/audit: Groceries 9.835 giỏ, 169 items;
Splice giữ 3.185 mẫu, window 25:36 và 48 IDs, nhãn EI/IE/N giữ riêng.

Bước tiếp theo là nối dữ liệu với miner/rules, xuất top-10 luật Groceries và
diễn giải; khai thác patterns DNA và so sánh support trên cả ba lớp. Với candidate
infrequent ở lớp khác, đếm lại support từ observations thay vì gán zero.
RUN_APPLICATIONS chỉ bật khi pipeline khai thác cuối được triển khai và kiểm chứng.


In [ ]:
if RUN_OPTIMIZED:
    run_julia("scripts/mine.jl", "--input", "data/toy/basic.txt", "--output", NOTEBOOK_OUTPUT / "bitset.txt",
              "--minsup", "40%", "--backend", "bitset")
    run_julia("scripts/compare_spmf.jl", NOTEBOOK_OUTPUT / "bitset.txt", "tests/reference/spmf/basic.txt")
else:
    print("RUN_OPTIMIZED đang tắt; full suite đã kiểm cả hai backend.")

if RUN_APPLICATIONS:
    raise NotImplementedError("Preprocessing và miner/rules đã có. Cần nối pipeline ứng dụng, xuất kết quả mined thật và diễn giải trước khi bật RUN_APPLICATIONS.")
else:
    print("Preprocessing đã có; pipeline ứng dụng mined và diễn giải là bước tiếp theo.")


## Benchmark A

Controller đã triển khai, hỗ trợ Linux/macOS; RUN_BENCHMARKS mặc định tắt.
Nếu bật, dùng output mới để giữ nguyên evidence đã publish. Hai phạm vi timing
được lưu riêng; allocation không thay peak RSS. Xem docs/experiments.md.


In [ ]:
if RUN_BENCHMARKS:
    # Real A pipeline; Java >=21, Python >=3.11. Full grid can take several minutes.
    subprocess.run([sys.executable, "scripts/prepare_spmf.py", "--java", os.environ.get("ECLAT_JAVA", "java")], check=True)
    subprocess.run([sys.executable, "scripts/acquire_benchmarks.py"], check=True)
    run_julia("scripts/benchmark_runner.jl", "--java", os.environ.get("ECLAT_JAVA", "java"), "--output", str(NOTEBOOK_OUTPUT / "benchmark"))
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-plot.txt"], check=True)
    subprocess.run([sys.executable, "scripts/plot_results.py", "--results", str(NOTEBOOK_OUTPUT / "benchmark")], check=True)
else:
    print("Benchmark pipeline đã có; RUN_BENCHMARKS đang tắt.")


## Final cold-run — B chủ trì, A review

Trước nộp: bỏ mọi TODO cần thực thi, bật đủ cờ, restart runtime và Run all.
Đặt FINAL_DELIVERY_VALIDATION=True để kiểm artifacts cơ bản; vẫn cần reviewer kiểm
chất lượng nội dung, đủ đồ thị, đúng metrics và public links. Report phải 10–15
trang IEEE tiếng Việt, paper gốc + ≥3 nguồn khác. README phải có public Colab URL.

In [ ]:
if FINAL_DELIVERY_VALIDATION:
    assert all([RUN_MINER, RUN_SPMF, RUN_OPTIMIZED, RUN_APPLICATIONS, RUN_BENCHMARKS]), "Chưa bật đủ pipeline."
    run_julia("tests/runtests.jl", env={"ECLAT_TEST_MINER": "1", "ECLAT_TEST_SPMF": "1",
                                      "ECLAT_TEST_BACKENDS": "list,bitset"})
    for relative in ["docs/Report.pdf", "results/applications/groceries_rules.csv",
                     "results/applications/dna_patterns.csv"]:
        assert (ROOT / relative).is_file(), f"Thiếu artifact: {relative}"
    print("Đã qua kiểm tra artifacts cơ bản; A/B tiếp tục rà checklist DELIVERY.md.")
else:
    print("Chưa nghiệm thu cuối: cần kết quả ứng dụng mined, Report.pdf và remote Colab cold-run.")